<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 14 · Oanda Demo Accounts and EURUSD Diagnostics

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook reflects the structure of the Oanda diagnostics without requiring live credentials.

- Review the candle schema expected from the Oanda client.
- Apply `compute_simple_returns` on synthetic candles.
- Reproduce the price-path and return-histogram visualisation.


In [ ]:
import sys
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/yhilpisch/paatcode.git"
    ROOT_DIR = Path("/content/paatcode")
    if not ROOT_DIR.exists():
        subprocess.run([
            "git",
            "clone",
            REPO_URL,
            str(ROOT_DIR),
        ], check=True)
    PROJECT_ROOT = ROOT_DIR
else:
    PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch14_oanda_eurusd_eod as ch14  # Oanda diagnostics helpers


### 1. Synthetic Candle Data

We mimic the candle structure returned by the Oanda wrapper so that all downstream code paths remain realistic even without a network connection.

In [ ]:
rng = np.random.default_rng(seed=14)
dates = pd.date_range("2024-01-01", periods=260, freq="B")
price = 1.10 + 0.02 * np.cumsum(rng.normal(0.0, 0.01, size=dates.shape[0]))

candles = pd.DataFrame(
    {
        "o": price * (1.0 + rng.normal(0.0, 0.0005, size=price.shape[0])),
        "h": price * (1.0 + 0.0015),
        "l": price * (1.0 - 0.0015),
        "c": price,
        "volume": rng.integers(1_000, 5_000, size=price.shape[0]),
    },
    index=dates,
)
candles.head()


### 2. Returns and Diagnostics

We now apply the same `compute_simple_returns` helper that is used in the live Oanda script and summarise basic statistics.

In [ ]:
rets = ch14.compute_simple_returns(candles["c"])
print(f"Number of observations: {rets.shape[0]}")
print(f"Mean daily return     : {rets.mean(): .5f}")
print(f"Daily volatility      : {rets.std(ddof=1): .5f}")


### 3. Price Path and Histogram

The figure mirrors the Chapter 14 diagnostic: a normalised price path and a histogram of daily returns.

In [ ]:
fig, (ax_price, ax_ret) = plt.subplots(2, 1, figsize=(7.0, 4.6))

normed = candles["c"] / candles["c"].iloc[0]
ax_price.plot(normed.index, normed.values, color="tab:blue")
ax_price.set_ylabel("price (normalised)")
ax_price.set_title("EURUSD: synthetic daily mid-price diagnostics")
ax_price.grid(True, alpha=0.3)

ax_ret.hist(rets.values, bins=60, color="tab:orange", alpha=0.8)
ax_ret.set_xlabel("daily simple return")
ax_ret.set_ylabel("frequency")
ax_ret.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


### Takeaways

- The Oanda wrapper adds network access but the downstream diagnostics are based on plain pandas objects.
- Practising with synthetic candles helps you debug logic before involving external APIs.
- When credentials are configured, you can swap the synthetic candles for real ones and keep the rest of this notebook unchanged.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">